<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Build step 2</div>
<div style="font-size:24px;font-weight:700;">&#129352; Silver: cleaned, governed, related</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Dimensions and facts: types cast, store IDs upper-cased, duplicate re-sends removed, JSON line items exploded.</p>


In [0]:
# Parameters come from the bundle's build job: the catalog and the three schemas the bundle created.
# This sets USE CATALOG and the session variables bronze_schema, silver_schema and gold_schema.
from bean_and_brew import init, fill

cfg = init(spark, dbutils)


In [0]:
%sql
-- Silver dimensions: typed, trimmed, IDs and emails normalized
CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_store')
COMMENT 'Conformed store dimension for Bean & Brew Coffee Co. One row per store location with typed attributes. Source: bronze.raw_stores.'
AS SELECT upper(trim(store_id)) AS store_id, store_name, city, state, region, store_format,
  to_date(open_date) AS open_date, cast(square_feet AS INT) AS square_feet, cast(seating_capacity AS INT) AS seating_capacity,
  store_manager, cast(latitude AS DOUBLE) latitude, cast(longitude AS DOUBLE) longitude
FROM IDENTIFIER(bronze_schema || '.raw_stores');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_product')
COMMENT 'Conformed product/menu dimension. One row per sellable SKU with 2025 list price, standard unit cost, and standard margin. Source: bronze.raw_products.'
AS SELECT product_id, product_name, category, temperature AS serving_type,
  cast(list_price AS DECIMAL(10,2)) AS list_price_2025,
  cast(round(list_price * 1.05, 2) AS DECIMAL(10,2)) AS list_price_2026,
  cast(unit_cost AS DECIMAL(10,2)) AS unit_cost,
  cast(round((list_price - unit_cost) / list_price, 4) AS DECIMAL(6,4)) AS standard_margin_pct,
  cast(is_seasonal AS BOOLEAN) AS is_seasonal,
  category IN ('Espresso Drinks','Brewed Coffee','Cold Drinks','Tea','Seasonal') AS is_beverage
FROM IDENTIFIER(bronze_schema || '.raw_products');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.dim_loyalty_member')
COMMENT 'Cleaned Bean & Brew Rewards loyalty member dimension. Emails lower-cased, home store IDs normalized, dates typed. Source: bronze.raw_loyalty_members.'
AS SELECT member_id, first_name, last_name, lower(trim(email)) AS email, to_date(signup_date) AS signup_date,
  upper(home_store_id) AS home_store_id, tier AS loyalty_tier, birth_month
FROM IDENTIFIER(bronze_schema || '.raw_loyalty_members');

In [0]:
%sql
-- Silver facts: duplicates removed, voids excluded from line items, JSON exploded, price book and promos applied
CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_sales_line_items')
COMMENT 'Cleaned POS line-item fact. One row per completed ticket line. Duplicate POS re-sends removed, voided tickets excluded, JSON payload exploded, prices resolved from the active price book, promo discounts applied. Sources: bronze.raw_pos_transactions, silver.dim_product.'
AS
WITH dedup AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, row_number() OVER (PARTITION BY transaction_id ORDER BY _ingested_at DESC) rn FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions')) WHERE rn = 1),
exploded AS (
  SELECT d.transaction_id, upper(d.store_id) AS store_id, to_timestamp(d.transaction_ts) AS transaction_ts,
    d.order_channel, d.loyalty_member_id, d.promo_code, d.price_book, li.line, li.sku, li.qty
  FROM dedup d
  LATERAL VIEW explode(from_json(d.line_items_json, 'array<struct<line:int,sku:string,qty:int>>')) e AS li
  WHERE d.ticket_status = 'COMPLETED'),
priced AS (
  SELECT
    concat(e.transaction_id, '-', e.line) AS line_item_id,
    e.transaction_id, e.line AS line_number, e.store_id, e.transaction_ts,
    to_date(e.transaction_ts) AS business_date, hour(e.transaction_ts) AS business_hour,
    e.order_channel, e.loyalty_member_id, e.promo_code,
    e.sku AS product_id, e.qty AS quantity,
    CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END AS unit_price,
    cast(e.qty * CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END AS DECIMAL(12,2)) AS gross_sales,
    cast(round(e.qty * CASE WHEN e.price_book = 'menu_v2026' THEN p.list_price_2026 ELSE p.list_price_2025 END *
      CASE WHEN e.promo_code = 'LOYALTY10' THEN 0.10 WHEN e.promo_code = 'HAPPYHOUR' AND p.is_beverage THEN 0.20 ELSE 0 END, 2) AS DECIMAL(12,2)) AS discount_amount,
    cast(e.qty * p.unit_cost AS DECIMAL(12,2)) AS cost_of_goods
  FROM exploded e JOIN IDENTIFIER(silver_schema || '.dim_product') p ON e.sku = p.product_id)
SELECT *, cast(gross_sales - discount_amount AS DECIMAL(12,2)) AS net_sales FROM priced;

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_transactions')
COMMENT 'Cleaned POS ticket (transaction header) fact. One row per unique ticket, including voided tickets (flagged). Totals rolled up from line items; sales tax applied by store state. Sources: bronze.raw_pos_transactions, silver.fact_sales_line_items, silver.dim_store.'
AS
WITH dedup AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, row_number() OVER (PARTITION BY transaction_id ORDER BY _ingested_at DESC) rn FROM IDENTIFIER(bronze_schema || '.raw_pos_transactions')) WHERE rn = 1),
items AS (
  SELECT transaction_id, count(*) line_count, sum(quantity) item_qty, sum(gross_sales) gross_sales, sum(discount_amount) discount_amount,
    sum(net_sales) net_sales, sum(cost_of_goods) cost_of_goods
  FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') GROUP BY transaction_id)
SELECT d.transaction_id, upper(d.store_id) AS store_id, d.register_id, to_timestamp(d.transaction_ts) AS transaction_ts,
  to_date(to_timestamp(d.transaction_ts)) AS business_date, hour(to_timestamp(d.transaction_ts)) AS business_hour,
  d.order_channel, d.loyalty_member_id, d.loyalty_member_id IS NOT NULL AS is_loyalty_transaction,
  d.payment_method, d.promo_code, d.ticket_status = 'VOIDED' AS is_voided,
  coalesce(i.line_count,0) line_count, coalesce(i.item_qty,0) item_qty,
  coalesce(i.gross_sales,0) gross_sales, coalesce(i.discount_amount,0) discount_amount, coalesce(i.net_sales,0) net_sales,
  cast(round(coalesce(i.net_sales,0) * CASE s.state WHEN 'WA' THEN 0.1025 WHEN 'OR' THEN 0 WHEN 'CA' THEN 0.0875 WHEN 'AZ' THEN 0.086 WHEN 'TX' THEN 0.0825 ELSE 0.0881 END, 2) AS DECIMAL(12,2)) AS sales_tax,
  coalesce(i.cost_of_goods,0) cost_of_goods
FROM dedup d LEFT JOIN items i USING (transaction_id) JOIN IDENTIFIER(silver_schema || '.dim_store') s ON upper(d.store_id) = s.store_id;

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_inventory_daily')
COMMENT 'Cleaned end-of-day inventory position per store and stock item, with valuation and reorder flag. Source: bronze.raw_inventory_counts.'
AS SELECT upper(store_id) store_id, count_date, item_code, item_name, unit_of_measure,
  cast(on_hand_qty AS DECIMAL(12,1)) on_hand_qty, cast(par_level AS DECIMAL(12,1)) par_level, cast(reorder_point AS DECIMAL(12,1)) reorder_point,
  cast(unit_cost AS DECIMAL(10,2)) unit_cost, cast(on_hand_qty * unit_cost AS DECIMAL(12,2)) AS inventory_value,
  on_hand_qty < reorder_point AS is_below_reorder_point
FROM IDENTIFIER(bronze_schema || '.raw_inventory_counts');

CREATE TABLE IF NOT EXISTS IDENTIFIER(silver_schema || '.fact_labor_daily')
COMMENT 'Cleaned daily labor fact per store from the WFM system, with computed labor cost. Source: bronze.raw_labor_shifts.'
AS SELECT upper(store_id) store_id, business_date, cast(scheduled_hours AS DECIMAL(8,1)) scheduled_hours, cast(worked_hours AS DECIMAL(8,1)) worked_hours,
  headcount, cast(avg_hourly_wage AS DECIMAL(8,2)) avg_hourly_wage, cast(worked_hours * avg_hourly_wage AS DECIMAL(12,2)) AS labor_cost
FROM IDENTIFIER(bronze_schema || '.raw_labor_shifts');